In [0]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def load_gold_table(spark: SparkSession, catalog: str, schema: str, table: str) -> DataFrame:
    """Load a Delta table from the specified catalog.schema.table."""
    return spark.table(f"{catalog}.{schema}.{table}")


def engineer_demand_features(df: DataFrame) -> DataFrame:
    """
    Add lag features, moving average, and calendar features to the demand DataFrame.

    Partitioned by product_name, ordered by transaction_date.
    """
    w = Window.partitionBy("product_name").orderBy("transaction_date")

    return (
        df.withColumn("demand_lag_1", F.lag("total_demand_quantity", 1).over(w))
          .withColumn("demand_lag_7", F.lag("total_demand_quantity", 7).over(w))
          .withColumn("demand_ma_7", F.avg("total_demand_quantity").over(w.rowsBetween(-6, 0)))
          .select(
              "transaction_date",
              "product_name",
              "destination_city",
              F.col("total_demand_quantity").alias("total_demand"),
              "avg_unit_price",
              F.col("total_available_inventory").alias("total_inventory"),
              "demand_lag_1",
              "demand_lag_7",
              "demand_ma_7",
              F.dayofweek("transaction_date").alias("day_of_week"),
              F.month("transaction_date").alias("month"),
          )
    )


def write_feature_table(df: DataFrame, catalog: str, schema: str, table: str) -> None:
    """Write the feature DataFrame to a Delta table (overwrite mode)."""
    df.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable(
        f"{catalog}.{schema}.{table}"
    )

In [0]:
catalog_name = "ct_oil_gas" 
gold_schema_name = "sc_gold"
gold_table_name = "daily_demand"
feature_table_name = "feature_demand"


In [0]:
df = load_gold_table(spark, catalog_name, gold_schema_name, gold_table_name)

# display(df.limit(10))

In [0]:

df_new = engineer_demand_features(df)

# display(df_new)

In [0]:
write_feature_table(df_new, catalog_name, gold_schema_name, feature_table_name)